# Validation of LLM Explanations: With vs Without XAI

This notebook validates the LLM-generated analyses from the comparison experiment.
It checks whether the models:
- Correctly rank features per class
- Cite accurate SHAP values
- Fabricate nonexistent misclassifications or statistics
- Improve their analysis when given XAI evidence (Chat A vs Chat B)

In [ ]:
import json
import re
import numpy as np
import pandas as pd
from IPython.display import display, Markdown, HTML

## 1. Ground Truth: Recompute Actual SHAP Values

In [ ]:
import shap
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import train_test_split
from imblearn.over_sampling import SMOTE
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score

df = pd.read_csv("./Network_logs.csv")
networkData = df.copy()
networkData.drop(['Source_IP', 'Destination_IP', 'Intrusion'], axis=1, inplace=True)

categorical_cols = ['Request_Type', 'Protocol', 'User_Agent', 'Status', 'Port']
for col in categorical_cols:
    networkData[col] = networkData[col].astype('category').cat.codes

target_encoder = LabelEncoder()
networkData['Scan_Type_Label'] = target_encoder.fit_transform(networkData['Scan_Type'])
networkData.drop(['Scan_Type'], axis=1, inplace=True)

scaler = StandardScaler()
networkData['Payload_Size'] = scaler.fit_transform(networkData[['Payload_Size']])

X = networkData.drop(['Scan_Type_Label'], axis=1)
y = networkData['Scan_Type_Label']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42, stratify=y)

smote = SMOTE()
X_train, y_train = smote.fit_resample(X_train, y_train)

model = RandomForestClassifier()
model.fit(X_train, y_train)
y_pred = model.predict(X_test)

feature_names = list(X.columns)
class_names = list(target_encoder.classes_)

np.random.seed(42)
sample_idx = np.random.choice(X_test.index, size=min(200, len(X_test)), replace=False)
X_sample = X_test.loc[sample_idx]

explainer = shap.TreeExplainer(model)
shap_values = explainer.shap_values(X_sample)

# Ground truth SHAP global
shap_ground_truth = {}
for cls_idx, cls_name in enumerate(class_names):
    mean_abs = np.abs(shap_values[:, :, cls_idx]).mean(axis=0)
    shap_ground_truth[cls_name] = {feat: round(float(val), 6) for feat, val in zip(feature_names, mean_abs)}

# Ground truth rankings (feature names sorted by importance, descending)
shap_rankings = {}
for cls_name in class_names:
    ranked = sorted(shap_ground_truth[cls_name].items(), key=lambda x: x[1], reverse=True)
    shap_rankings[cls_name] = [f[0] for f in ranked]

accuracy = round(accuracy_score(y_test, y_pred), 4)

# Count actual misclassifications in the 20-record sample
pred_sample = pd.DataFrame({"real": y_test, "predicted": y_pred})
pred_sample_20 = pred_sample.sample(20, random_state=42)
n_misclass = (pred_sample_20["real"] != pred_sample_20["predicted"]).sum()

print("Ground Truth SHAP Global Values:")
print(json.dumps(shap_ground_truth, indent=2))
print(f"\nGround Truth Rankings per Class:")
for cls, ranking in shap_rankings.items():
    print(f"  {cls}: {' > '.join(ranking)}")
print(f"\nModel Accuracy: {accuracy}")
print(f"Misclassifications in 20-record sample: {n_misclass}")

## 2. Load LLM Responses

In [ ]:
with open("resultados_comparison_with_without_xai.json", "r", encoding="utf-8") as f:
    results_comparison = json.load(f)

# Only the 3 models with complete results
MODEL_NAMES = ["glm-4.7-flash:latest", "qwen3:14b", "gpt-oss:20b"]

for m in MODEL_NAMES:
    a_len = len(results_comparison[m]["chat_a_without_xai"])
    b_len = len(results_comparison[m]["chat_b_with_xai"])
    print(f"{m}: Chat A = {a_len} chars, Chat B = {b_len} chars")

## 3. Pretty Print: All Responses Side by Side

In [ ]:
for model_name in MODEL_NAMES:
    chat_a = results_comparison[model_name]["chat_a_without_xai"]
    chat_b = results_comparison[model_name]["chat_b_with_xai"]

    display(Markdown(f"---\n# {model_name}\n---"))
    display(Markdown(f"## Chat A: Without XAI\n\n{chat_a}"))
    display(Markdown(f"## Chat B: With XAI (SHAP + LIME)\n\n{chat_b}"))

## 4. Validation: Feature Ranking Accuracy

For each model's Chat B response, extract the feature ranking per class and compare against ground truth.

In [ ]:
# Manually extracted rankings from Chat B responses (based on what each model stated)
# These are extracted from the actual text content above

chatb_rankings = {
    "glm-4.7-flash:latest": {
        "BotAttack": ["Port", "Payload_Size", "Status"],
        "Normal":    ["Port", "Payload_Size", "Status"],
        "PortScan":  ["Payload_Size", "Port", "Status"],
    },
    "qwen3:14b": {
        "BotAttack": ["Port", "Status", "Payload_Size"],
        "Normal":    ["Port", "Payload_Size", "Status"],
        "PortScan":  ["Payload_Size", "Port", "Status"],
    },
    "gpt-oss:20b": {
        "BotAttack": ["Port", "Status", "Payload_Size"],
        "Normal":    ["Port", "Status", "Payload_Size"],
        "PortScan":  ["Payload_Size", "Status", "Port"],
    },
}

# Compare top-3 ranking per class
print("=" * 70)
print("FEATURE RANKING VALIDATION (Chat B vs Ground Truth, Top 3)")
print("=" * 70)

ranking_scores = {}

for model_name in MODEL_NAMES:
    print(f"\n--- {model_name} ---")
    correct = 0
    total = 0
    for cls_name in class_names:
        gt_top3 = shap_rankings[cls_name][:3]
        pred_top3 = chatb_rankings[model_name][cls_name][:3]

        # Check exact position match
        matches = sum(1 for i in range(3) if gt_top3[i] == pred_top3[i])
        correct += matches
        total += 3

        status = "CORRECT" if gt_top3 == pred_top3 else "PARTIAL" if matches > 0 else "WRONG"
        print(f"  {cls_name}:")
        print(f"    Ground truth: {' > '.join(gt_top3)}")
        print(f"    Model stated: {' > '.join(pred_top3)}")
        print(f"    Position matches: {matches}/3 [{status}]")

    ranking_scores[model_name] = correct / total
    print(f"  Overall ranking accuracy: {correct}/{total} ({correct/total:.0%})")

## 5. Validation: SHAP Value Citation Accuracy

Extract numeric SHAP values cited in Chat B and compare to ground truth.

In [ ]:
# Manually extracted SHAP values cited in Chat B for BotAttack class (top 3 features)
# This is what each model explicitly wrote in their response

cited_shap = {
    "glm-4.7-flash:latest": {
        "BotAttack": {"Port": 0.23, "Payload_Size": 0.09, "Status": None},  # approximate, no exact values
        "Normal":    {"Port": 0.26, "Payload_Size": 0.27, "Status": None},
        "PortScan":  {"Payload_Size": 0.27, "Port": None, "Status": None},
    },
    "qwen3:14b": {
        "BotAttack": {"Port": 0.2372, "Status": 0.1378, "Payload_Size": 0.0927},
        "Normal":    {"Port": 0.2636, "Payload_Size": 0.1919, "Status": 0.1983},
        "PortScan":  {"Payload_Size": 0.269966, "Port": 0.0268, "Status": 0.060541},
    },
    "gpt-oss:20b": {
        "BotAttack": {"Port": 0.237, "Status": 0.138, "Payload_Size": 0.093},
        "Normal":    {"Port": 0.264, "Status": 0.198, "Payload_Size": 0.192},
        "PortScan":  {"Payload_Size": 0.270, "Port": 0.027, "Status": 0.061},
    },
}

print("=" * 70)
print("SHAP VALUE CITATION ACCURACY (Chat B vs Ground Truth)")
print("=" * 70)

citation_errors = {}

for model_name in MODEL_NAMES:
    print(f"\n--- {model_name} ---")
    errors = []
    for cls_name in class_names:
        print(f"  {cls_name}:")
        for feat, cited_val in cited_shap[model_name][cls_name].items():
            gt_val = shap_ground_truth[cls_name][feat]
            if cited_val is None:
                print(f"    {feat}: not cited (ground truth = {gt_val:.4f})")
                continue
            abs_err = abs(cited_val - gt_val)
            rel_err = abs_err / gt_val if gt_val > 0 else float('inf')
            errors.append(abs_err)
            status = "OK" if abs_err < 0.01 else "CLOSE" if abs_err < 0.02 else "OFF"
            print(f"    {feat}: cited={cited_val:.4f} vs actual={gt_val:.4f} (err={abs_err:.4f}) [{status}]")

    if errors:
        citation_errors[model_name] = {
            "mean_abs_error": np.mean(errors),
            "max_abs_error": np.max(errors),
            "n_cited": len(errors),
        }
        print(f"  Mean absolute error: {np.mean(errors):.4f}")
        print(f"  Max absolute error:  {np.max(errors):.4f}")

## 6. Validation: Fabrication Detection

Check if models invented misclassifications, fake percentages, or nonexistent patterns.

In [ ]:
fabrication_checks = {
    "glm-4.7-flash:latest": {
        "chat_a": {
            "claims_no_misclassifications": True,
            "fabricates_percentages": False,
            "fabricates_specific_shap_values": False,
            "notes": "Correctly identifies 0 misclassifications. Does not fabricate percentages. "
                     "Speculates on feature importance using cybersecurity intuition (reasonable for Chat A).",
        },
        "chat_b": {
            "claims_no_misclassifications": False,
            "fabricates_misclassifications": True,
            "fabricates_specific_shap_values": False,
            "notes": "FABRICATION: Invents a 'False Positive (Normal predicted as BotAttack)' and a "
                     "'False Negative (PortScan predicted as Normal)' that do NOT exist in the 20-record sample. "
                     "Also cites approximate SHAP values without exact numbers for most features.",
        },
    },
    "qwen3:14b": {
        "chat_a": {
            "claims_no_misclassifications": True,
            "fabricates_percentages": True,
            "fabricates_specific_shap_values": False,
            "notes": "FABRICATION: Invents feature importance percentages - 'User_Agent (72%)', "
                     "'Payload_Size (25%)', 'Port (8%)' etc. These numbers are completely made up. "
                     "Correctly identifies 0 misclassifications.",
        },
        "chat_b": {
            "claims_no_misclassifications": True,
            "fabricates_misclassifications": False,
            "fabricates_specific_shap_values": False,
            "notes": "Cites actual SHAP values from the provided data. Correctly acknowledges no misclassifications. "
                     "Minor issue: narrative text sometimes swaps ranking emphasis "
                     "(says 'Status is critical differentiator for BotAttack' when Port is actually #1).",
        },
    },
    "gpt-oss:20b": {
        "chat_a": {
            "claims_no_misclassifications": True,
            "fabricates_percentages": False,
            "fabricates_specific_shap_values": False,
            "notes": "Correctly identifies 0 misclassifications. Does not fabricate percentages. "
                     "Feature ranking is speculative but clearly marked as assumptions.",
        },
        "chat_b": {
            "claims_no_misclassifications": True,
            "fabricates_misclassifications": False,
            "fabricates_specific_shap_values": False,
            "notes": "Most faithful of the three. Cites near-exact SHAP values. Correctly states 0 misclassifications. "
                     "Appropriately caveats that 20-record sample is too small to trust.",
        },
    },
}

print("=" * 70)
print("FABRICATION DETECTION")
print("=" * 70)

for model_name in MODEL_NAMES:
    print(f"\n{'='*50}")
    print(f"{model_name}")
    print(f"{'='*50}")
    for phase in ["chat_a", "chat_b"]:
        phase_label = "Chat A (without XAI)" if phase == "chat_a" else "Chat B (with XAI)"
        checks = fabrication_checks[model_name][phase]
        has_fabrication = checks.get("fabricates_percentages", False) or checks.get("fabricates_misclassifications", False)
        status = "FABRICATION DETECTED" if has_fabrication else "CLEAN"
        print(f"\n  {phase_label}: [{status}]")
        print(f"    {checks['notes']}")

## 7. Validation: Chat A vs Chat B Improvement Assessment

Did XAI evidence actually improve the analysis?

In [ ]:
# Feature rankings stated in Chat A (without XAI) - extracted from text
chata_rankings = {
    "glm-4.7-flash:latest": {
        "BotAttack": ["Payload_Size", "User_Agent", "Port"],
        "Normal":    ["Payload_Size", "User_Agent", "Port"],
        "PortScan":  ["Payload_Size", "User_Agent", "Port"],
    },
    "qwen3:14b": {
        "BotAttack": ["User_Agent", "Payload_Size", "Port"],
        "Normal":    ["User_Agent", "Payload_Size", "Port"],
        "PortScan":  ["User_Agent", "Payload_Size", "Port"],
    },
    "gpt-oss:20b": {
        "BotAttack": ["Payload_Size", "Status", "Port"],
        "Normal":    ["Payload_Size", "Status", "Port"],
        "PortScan":  ["Payload_Size", "Status", "Port"],
    },
}

def ranking_accuracy(predicted_rankings, ground_truth_rankings):
    """Compute fraction of correct top-3 position matches across all classes."""
    correct = 0
    total = 0
    for cls_name in class_names:
        gt = ground_truth_rankings[cls_name][:3]
        pr = predicted_rankings[cls_name][:3]
        for i in range(3):
            if gt[i] == pr[i]:
                correct += 1
            total += 1
    return correct / total


print("=" * 70)
print("CHAT A vs CHAT B: RANKING IMPROVEMENT")
print("=" * 70)

improvement_data = []

for model_name in MODEL_NAMES:
    acc_a = ranking_accuracy(chata_rankings[model_name], shap_rankings)
    acc_b = ranking_accuracy(chatb_rankings[model_name], shap_rankings)
    delta = acc_b - acc_a
    improvement_data.append({
        "Model": model_name,
        "Chat A Ranking Acc": f"{acc_a:.0%}",
        "Chat B Ranking Acc": f"{acc_b:.0%}",
        "Improvement": f"+{delta:.0%}" if delta >= 0 else f"{delta:.0%}",
    })
    print(f"\n{model_name}:")
    print(f"  Chat A (no XAI):  {acc_a:.0%} ranking accuracy")
    print(f"  Chat B (with XAI): {acc_b:.0%} ranking accuracy")
    print(f"  Improvement: {'+' if delta >= 0 else ''}{delta:.0%}")

print("\n")
display(pd.DataFrame(improvement_data).set_index("Model"))

## 8. Summary Verdict

In [ ]:
verdict_md = f"""
# Validation Summary

**Ground Truth:**
- Model accuracy: **{accuracy}**
- Misclassifications in 20-record sample: **{n_misclass}**
- Top-3 feature rankings per class:
  - **BotAttack:** {' > '.join(shap_rankings['BotAttack'][:3])}
  - **Normal:** {' > '.join(shap_rankings['Normal'][:3])}
  - **PortScan:** {' > '.join(shap_rankings['PortScan'][:3])}

---

## Per-Model Verdict

### glm-4.7-flash
| Aspect | Chat A | Chat B |
|--------|--------|--------|
| Feature ranking | Wrong (guesses Payload_Size #1 for all) | Partially correct (gets PortScan right) |
| SHAP values cited | N/A | Approximate only, many missing |
| Fabrications | None | **Invents misclassifications** that don't exist |
| Overall | Reasonable speculation | Improved but unreliable due to fabricated errors |

### qwen3:14b
| Aspect | Chat A | Chat B |
|--------|--------|--------|
| Feature ranking | Wrong (guesses User_Agent #1 for all) | Mostly correct |
| SHAP values cited | **Fabricates percentages** (72%, 25%) | Cites actual SHAP values accurately |
| Fabrications | Fake importance percentages | Clean |
| Overall | Overconfident speculation | Strong improvement with XAI grounding |

### gpt-oss:20b
| Aspect | Chat A | Chat B |
|--------|--------|--------|
| Feature ranking | Partially right (gets Status importance) | Most accurate of all three |
| SHAP values cited | N/A | Near-exact values cited |
| Fabrications | None | None |
| Overall | Best Chat A speculation | Best Chat B faithfulness |

---

## Key Findings

1. **Without XAI (Chat A):** All models rely on cybersecurity heuristics and guess wrong
   about the actual model behavior. Common errors:
   - Overestimate User_Agent importance (actual SHAP < 0.006 for all classes)
   - Underestimate Port importance (actual #1 for BotAttack and Normal)
   - qwen3:14b fabricates confidence percentages

2. **With XAI (Chat B):** All models significantly improve feature ranking accuracy.
   XAI evidence successfully grounds the analysis in actual model behavior.

3. **Faithfulness varies:** gpt-oss:20b is the most faithful (cites exact values, no fabrications).
   glm-4.7-flash is the least reliable (invents misclassifications in Chat B).

4. **XAI corrects the biggest blind spot:** All Chat A responses overvalue User_Agent
   (cybersecurity intuition says scanners are identified by tool signatures). SHAP reveals
   the model actually relies on Port and Payload_Size, with User_Agent being nearly irrelevant
   (SHAP < 0.006). Chat B responses correctly drop User_Agent from the top features.
"""

display(Markdown(verdict_md))